# Bone Fracture Detection — YOLOv8l (3 Datasets, 9 Classes)

| Dataset | Key paths | Covers |
|---------|-----------|--------|
| archive/BoneFractureYolo8 | train/ valid/ test/ | elbow finger forearm humerus shoulder wrist |
| fracAtlasDataSet/FracAtlas | Annotations/YOLO/ + Utilities/dataset.csv + images/ | hand hip leg shoulder |
| graz-yolo-ready-v11 | images/ + labels/ + data.yaml | wrist (pediatric) |

9 unified classes: 0=elbow 1=finger 2=forearm 3=humerus 4=shoulder 5=wrist 6=hand 7=hip 8=leg

## Phase 0 — Environment & Dataset Discovery

In [ ]:
import subprocess, torch
from pathlib import Path

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected! Enable GPU in Kaggle settings.")

num_gpus = torch.cuda.device_count()
for i in range(num_gpus):
    p  = torch.cuda.get_device_properties(i)
    fg = torch.cuda.mem_get_info(i)[0] / 1e9
    tg = torch.cuda.mem_get_info(i)[1] / 1e9
    print(f"  GPU {i}: {p.name} | {tg:.1f} GB total | {fg:.1f} GB free")

DEVICE       = "0,1" if num_gpus >= 2 else "0"
KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR  = Path("/kaggle/working")
print(f"Device: cuda:{DEVICE}")

# Print top-3-level layout only (never recurse into image/annotation folders)
SKIP = {
    "images","labels","train","valid","val","test",
    "images_part1","images_part2","images_part3","images_part4",
    "Fractured","Non-Fractured","yolov5","supervisely","wrist",
    "pascalvoc","PASCAL VOC","VGG JSON","COCO","notebooks",
    "Augmentation","Splits","Fracture Split",
}
print("\n/kaggle/input/ layout:")
for a in sorted(KAGGLE_INPUT.iterdir()):
    if not a.is_dir(): continue
    print(f"  {a.name}/")
    for b in sorted(a.iterdir()):
        if not b.is_dir(): continue
        print(f"    {b.name}/")
        for c in sorted(b.iterdir()):
            if c.is_dir() and c.name not in SKIP:
                print(f"      {c.name}/")
            elif c.is_file() and c.suffix in {".yaml", ".csv", ".txt"}:
                print(f"      {c.name}")

# Step 1: try confirmed exact paths from your Kaggle filesystem
CANDIDATES = {
    "BONE8": [
        KAGGLE_INPUT/"datasets"/"atishayjain2204"/"archive"/"BoneFractureYolo8",
        KAGGLE_INPUT/"archive"/"BoneFractureYolo8",
    ],
    "FA": [
        KAGGLE_INPUT/"datasets"/"atishayjain2204"/"fracatlasdataset"/"FracAtlas",
        KAGGLE_INPUT/"fracatlasdataset"/"FracAtlas",
        KAGGLE_INPUT/"fracAtlasDataSet"/"FracAtlas",
    ],
    "GRAZ": [
        KAGGLE_INPUT/"datasets"/"avizith"/"graz-yolo-ready-v11",
        KAGGLE_INPUT/"graz-yolo-ready-v11",
    ],
}

def pick(key):
    for p in CANDIDATES[key]:
        if p.exists():
            return p
    return None

BONE8_DIR = pick("BONE8")
FA_DIR    = pick("FA")
GRAZ_DIR  = pick("GRAZ")

# Step 2: fast fallback search (dirs only, skips heavy folders)
SKIP_WALK = {
    "images","labels","train","valid","val","test",
    "images_part1","images_part2","images_part3","images_part4",
    "Fractured","Non-Fractured","yolov5","YOLO","yolo",
    "supervisely","wrist","pascalvoc","PASCAL VOC","VGG JSON","COCO",
    "notebooks","Augmentation","Splits","Fracture Split",
    "bone fracture detection.v4-v4.yolov8",
}

def fast_find(keyword, need_dir=None, need_file=None):
    kw = keyword.lower()
    found = []
    def _w(p, d):
        if d > 7: return
        try:
            for c in p.iterdir():
                if not c.is_dir() or c.name in SKIP_WALK: continue
                if kw in c.name.lower():
                    ok = True
                    if need_dir  and not (c/need_dir).is_dir():   ok = False
                    if need_file and not (c/need_file).is_file():
                        ok = any((s/need_file).is_file()
                                 for s in c.iterdir()
                                 if s.is_dir() and s.name not in SKIP_WALK)
                    if ok: found.append(c)
                _w(c, d+1)
        except PermissionError: pass
    _w(KAGGLE_INPUT, 0)
    return sorted(found, key=lambda x: len(x.parts))[0] if found else None

if BONE8_DIR is None:
    print("Searching BoneFractureYolo8 ...")
    BONE8_DIR = fast_find("BoneFractureYolo8", need_dir="train")
if FA_DIR is None:
    print("Searching FracAtlas ...")
    FA_DIR = fast_find("FracAtlas", need_dir="Annotations")
if GRAZ_DIR is None:
    print("Searching graz-yolo-ready ...")
    GRAZ_DIR = fast_find("graz-yolo-ready", need_file="data.yaml")
    if GRAZ_DIR is None:
        GRAZ_DIR = fast_find("graz", need_file="data.yaml")

print("\nDataset paths:")
all_ok = True
for label, path in [("BoneFractureYolo8", BONE8_DIR),
                    ("FracAtlas",         FA_DIR),
                    ("graz-yolo-ready",   GRAZ_DIR)]:
    ok = path is not None and path.exists()
    if not ok: all_ok = False
    print(f"  [{'OK' if ok else 'MISSING'}] {label:20s}: {path}")
if not all_ok:
    raise FileNotFoundError("Dataset(s) missing. See layout above.")
print("All 3 datasets found. Ready.")


## Phase 1 — Install Dependencies

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics>=8.2.0"], check=True)

import os
import shutil
import random
import yaml
import csv
from pathlib import Path
from collections import Counter
print("Imports ready.")


## Phase 2 — Merge 3 Datasets (9 unified classes)

In [ ]:
# 9-class unified schema
UNIFIED_CLASSES = [
    "elbow_fracture",    # 0  BoneFractureYolo8 class 0
    "finger_fracture",   # 1  BoneFractureYolo8 class 1
    "forearm_fracture",  # 2  BoneFractureYolo8 class 2
    "humerus_fracture",  # 3  BoneFractureYolo8 class 3 + 4
    "shoulder_fracture", # 4  BoneFractureYolo8 class 5  + FracAtlas Shoulder
    "wrist_fracture",    # 5  BoneFractureYolo8 class 6  + GRAZ fracture
    "hand_fracture",     # 6  FracAtlas Hand
    "hip_fracture",      # 7  FracAtlas Hip
    "leg_fracture",      # 8  FracAtlas Leg
]
NC = len(UNIFIED_CLASSES)

BONE8_REMAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3, 5: 4, 6: 5}
FA_BODYPART = {"Hand": 6, "Hip": 7, "Leg": 8, "Shoulder": 4}

GRAZ_YAML     = yaml.safe_load((GRAZ_DIR / "data.yaml").read_text())
graz_names    = GRAZ_YAML.get("names", [])
GRAZ_FRAC_CLS = next(
    (i for i, n in enumerate(graz_names) if "fracture" in str(n).lower()), 3
)
GRAZ_TGT_CLS = 5   # -> wrist_fracture
print(f"GRAZ classes : {graz_names}")
print(f"GRAZ fracture class id : {GRAZ_FRAC_CLS} -> unified {GRAZ_TGT_CLS} (wrist_fracture)")

MERGED_DIR  = WORKING_DIR / "merged_dataset"
MERGED_YAML = WORKING_DIR / "merged_data.yaml"
random.seed(42)

for sp in ["train", "val", "test"]:
    (MERGED_DIR / sp / "images").mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / sp / "labels").mkdir(parents=True, exist_ok=True)

counter = Counter()

def get_images(d: Path):
    imgs = []
    for ext in ["*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG"]:
        imgs.extend(d.glob(ext))
    return imgs

def add_sample(img: Path, lines: list, split: str, prefix: str):
    stem = f"{prefix}_{img.stem}"
    shutil.copy2(img, MERGED_DIR / split / "images" / f"{stem}{img.suffix}")
    (MERGED_DIR / split / "labels" / f"{stem}.txt").write_text("\n".join(lines))
    for line in lines:
        counter[UNIFIED_CLASSES[int(line.split()[0])]] += 1

def remap(lbl: Path, remap_dict: dict) -> list:
    if not lbl or not lbl.exists():
        return []
    out = []
    for line in lbl.read_text().strip().splitlines():
        parts = line.strip().split()
        if parts and int(parts[0]) in remap_dict:
            parts[0] = str(remap_dict[int(parts[0])])
            out.append(" ".join(parts))
    return out

def keep_cls(lbl: Path, keep: int, new_cls: int) -> list:
    if not lbl or not lbl.exists():
        return []
    out = []
    for line in lbl.read_text().strip().splitlines():
        parts = line.strip().split()
        if parts and int(parts[0]) == keep:
            parts[0] = str(new_cls)
            out.append(" ".join(parts))
    return out

print("Config ready.")


In [ ]:
# DATASET 1: BoneFractureYolo8
print("=" * 60)
print(f"BoneFractureYolo8: {BONE8_DIR}")

total_b8 = 0
for src_sp in ["train", "valid", "test"]:
    dst_sp  = "val" if src_sp == "valid" else src_sp
    img_dir = BONE8_DIR / src_sp / "images"
    lbl_dir = BONE8_DIR / src_sp / "labels"
    if not img_dir.exists():
        print(f"  {src_sp}: no images/ found, skipping")
        continue
    imgs = get_images(img_dir)
    for img in imgs:
        lines = remap(lbl_dir / f"{img.stem}.txt", BONE8_REMAP)
        add_sample(img, lines, dst_sp, "b8")
    total_b8 += len(imgs)
    print(f"  {src_sp:6s} -> {dst_sp}: {len(imgs):,} images")

print(f"  BoneFractureYolo8 TOTAL: {total_b8:,}")


In [ ]:
# DATASET 2: FracAtlas
# FracAtlas directory structure:
#   images/Fractured/Hand/      -> hand_fracture    (cls 6)
#   images/Fractured/Hip/       -> hip_fracture     (cls 7)
#   images/Fractured/Leg/       -> leg_fracture     (cls 8)
#   images/Fractured/Shoulder/  -> shoulder_fracture (cls 4)
# YOLO labels: Annotations/YOLO/{stem}.txt  (class 0 = fractured)
# No dataset.csv needed - body_part is inferred from subdirectory name.
print("=" * 60)
print(f"FracAtlas: {FA_DIR}")

# -- Find YOLO label directory --
YOLO_DIR = None
for cand in [FA_DIR/"Annotations"/"YOLO",
             FA_DIR/"Annotations"/"yolo",
             FA_DIR/"annotations"/"YOLO"]:
    if cand.exists() and any(cand.glob("*.txt")):
        YOLO_DIR = cand
        break
if YOLO_DIR is None:
    for d in FA_DIR.rglob("*"):
        if d.is_dir() and d.name.upper() == "YOLO" and any(d.glob("*.txt")):
            YOLO_DIR = d
            break
assert YOLO_DIR, f"YOLO label dir not found inside {FA_DIR}"
print(f"  YOLO labels : {YOLO_DIR}")

# -- Collect fractured images from directory structure --
# Primary: images/Fractured/{BodyPart}/  -- gives body_part directly
# Fallback: images/Fractured/            -- flat, mark as shoulder (most common)
fractured_root = FA_DIR / "images" / "Fractured"
assert fractured_root.exists(), f"images/Fractured/ not found in {FA_DIR}"

samples = []
skipped = Counter()

# Check if body_part subdirs exist (Hand, Hip, Leg, Shoulder)
body_part_dirs = [d for d in fractured_root.iterdir()
                  if d.is_dir() and d.name in FA_BODYPART]

if body_part_dirs:
    print(f"  Layout: per-body-part subdirs: {[d.name for d in body_part_dirs]}")
    for bp_dir in body_part_dirs:
        bp_name = bp_dir.name                    # "Hand", "Hip", "Leg", "Shoulder"
        cls     = FA_BODYPART[bp_name]
        imgs    = get_images(bp_dir)
        for img in imgs:
            lbl = YOLO_DIR / f"{img.stem}.txt"
            samples.append((img, cls, lbl))
        print(f"    {bp_name:10s} (cls {cls}): {len(imgs):,} images")
else:
    # Flat layout: all fractured images in one folder
    # Try to read body_part from any CSV we can find
    print("  Layout: flat (no body_part subdirs) — scanning for metadata CSV ...")
    flat_imgs = get_images(fractured_root)
    print(f"  Flat images: {len(flat_imgs):,}")

    # Try every CSV anywhere under FA_DIR or its parent
    meta = {}
    for search_root in [FA_DIR, FA_DIR.parent]:
        for csv_file in search_root.rglob("*.csv"):
            try:
                with open(csv_file, newline="", encoding="utf-8") as f:
                    reader = csv.DictReader(f)
                    cols = reader.fieldnames or []
                    # Only use if has body_part info
                    bp_col = next((c for c in cols
                                   if "body" in c.lower() or "anatomy" in c.lower()
                                   or "label" in c.lower()), None)
                    frac_col = next((c for c in cols
                                     if "frac" in c.lower()), None)
                    id_col   = next((c for c in cols
                                     if "image" in c.lower() or c.lower() == "id"
                                     or "file" in c.lower()), None)
                    if bp_col and id_col:
                        print(f"  Using CSV: {csv_file}")
                        print(f"    id_col={id_col!r} bp_col={bp_col!r} frac_col={frac_col!r}")
                        for row in reader:
                            img_id = Path(row[id_col].strip()).stem
                            frac   = str(row.get(frac_col, "1")).strip() if frac_col else "1"
                            bp     = row[bp_col].strip()
                            if img_id:
                                meta[img_id] = {
                                    "fractured": frac in ("1","True","true","yes",""),
                                    "body_part": bp,
                                }
                        break
            except Exception:
                continue
        if meta:
            break

    if meta:
        for img in flat_imgs:
            m = meta.get(img.stem)
            if not m:                            skipped["no_csv"]    += 1; continue
            if not m["fractured"]:               skipped["not_frac"]  += 1; continue
            bp  = m["body_part"].strip()
            cls = FA_BODYPART.get(bp) or FA_BODYPART.get(bp.capitalize())
            if cls is None:                      skipped["unknown_bp"] += 1; continue
            samples.append((img, cls, YOLO_DIR / f"{img.stem}.txt"))
        print(f"  With CSV: {len(samples):,} usable  skipped: {dict(skipped)}")
    else:
        # Last resort: no body_part info at all — include all as shoulder_fracture
        # (most common anatomy in FracAtlas)
        print("  WARNING: No body_part metadata found.")
        print("  Using ALL fractured images as shoulder_fracture (class 4) fallback.")
        for img in flat_imgs:
            samples.append((img, 4, YOLO_DIR / f"{img.stem}.txt"))

print(f"  Total usable: {len(samples):,}")
if len(samples) == 0:
    print("  ERROR: 0 samples. Check dataset structure above.")

# -- 75/15/10 split --
random.shuffle(samples)
n = len(samples); nt = int(n*0.75); nv = int(n*0.15)
for sp, subset in [("train", samples[:nt]),
                   ("val",   samples[nt:nt+nv]),
                   ("test",  samples[nt+nv:])]:
    for img, cls, lbl in subset:
        lines = remap(lbl, {0: cls})
        add_sample(img, lines, sp, "fa")
    print(f"  {sp:5s}: {len(subset):,}")
print(f"  FracAtlas TOTAL: {n:,}")


In [ ]:
# DATASET 3: GRAZPEDWRI-DX (graz-yolo-ready-v11)
# Keep ONLY fracture class -> unified class 5 (wrist_fracture)
print("=" * 60)
print(f"GRAZPEDWRI-DX: {GRAZ_DIR}")

img_base = GRAZ_DIR / "images"
lbl_base = GRAZ_DIR / "labels"
assert img_base.exists(), f"images/ not found in {GRAZ_DIR}"
assert lbl_base.exists(), f"labels/ not found in {GRAZ_DIR}"

split_subs = [s for s in ["train","valid","val","test"] if (img_base/s).is_dir()]
total_gz = 0

if split_subs:
    print(f"  Layout: pre-split {split_subs}")
    for src_sp in split_subs:
        dst_sp = "val" if src_sp in ("valid","val") else src_sp
        imgs   = get_images(img_base / src_sp)
        kept   = 0
        for img in imgs:
            lines = keep_cls(lbl_base/src_sp/f"{img.stem}.txt",
                             GRAZ_FRAC_CLS, GRAZ_TGT_CLS)
            add_sample(img, lines, dst_sp, "gz")
            if lines: kept += 1
        total_gz += len(imgs)
        print(f"  {src_sp} -> {dst_sp}: {len(imgs):,} imgs | {kept:,} w/ fracture")
else:
    print("  Layout: flat (random 75/15/10 split)")
    all_gz = get_images(img_base)
    print(f"  Total: {len(all_gz):,}")
    random.shuffle(all_gz)
    n = len(all_gz); nt = int(n*0.75); nv = int(n*0.15)
    for sp, subset in [("train",all_gz[:nt]),("val",all_gz[nt:nt+nv]),("test",all_gz[nt+nv:])]:
        kept = 0
        for img in subset:
            lines = keep_cls(lbl_base/f"{img.stem}.txt", GRAZ_FRAC_CLS, GRAZ_TGT_CLS)
            add_sample(img, lines, sp, "gz")
            if lines: kept += 1
        total_gz += len(subset)
        print(f"  {sp:5s}: {len(subset):,} imgs | {kept:,} w/ fracture")

print(f"  GRAZ TOTAL: {total_gz:,}")


In [ ]:
import yaml

cfg = {
    "path" : str(MERGED_DIR),
    "train": "train/images",
    "val"  : "val/images",
    "test" : "test/images",
    "nc"   : NC,
    "names": UNIFIED_CLASSES,
}
MERGED_YAML.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False))
print("merged_data.yaml:")
print(MERGED_YAML.read_text())

print("=" * 60)
print("MERGED DATASET SUMMARY")
grand = 0
for sp in ["train", "val", "test"]:
    n = len(list((MERGED_DIR/sp/"images").glob("*")))
    grand += n
    print(f"  {sp:5s}: {n:,} images")
print(f"  TOTAL: {grand:,}")

print()
print("Annotations per class:")
mx = max(counter.values(), default=1)
for cls in UNIFIED_CLASSES:
    cnt  = counter.get(cls, 0)
    bar  = chr(9608) * int(35 * cnt / mx)
    warn = "  << SPARSE" if cnt < 200 else ""
    print(f"  {cls:20s}: {cnt:6,}  {bar}{warn}")
print(f"Total annotations: {sum(counter.values()):,}")
print("Merge complete!")


## Phase 3 — Train YOLOv8l (300 epochs, AdamW, imgsz=800)

In [ ]:
from ultralytics import YOLO
import torch
from pathlib import Path

SEED=42; torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL="yolov8l.pt"; EPOCHS=300; IMGSZ=800; PATIENCE=50; RUN="fracture_v3"
LAST_PT = WORKING_DIR / "runs" / RUN / "weights" / "last.pt"

print(f"Model   : {MODEL}")
print(f"Data    : {MERGED_YAML}")
print(f"Device  : cuda:{DEVICE}")
print(f"Epochs  : {EPOCHS}  Patience: {PATIENCE}  ImgSz: {IMGSZ}")

if LAST_PT.exists():
    print(f"\n[INFO] Found {LAST_PT}, RESUMING training from last checkpoint!")
    model = YOLO(str(LAST_PT))
    results = model.train(resume=True)
else:
    print("\n[INFO] Starting fresh training...")
    model = YOLO(MODEL)
    results = model.train(
        data=str(MERGED_YAML), epochs=EPOCHS, imgsz=IMGSZ,
        batch=16, device=DEVICE,
        project=str(WORKING_DIR/"runs"), name=RUN, exist_ok=True,
        save=True, save_period=10, patience=PATIENCE,
        seed=SEED, plots=True, amp=True,
        optimizer="AdamW", lr0=0.001, lrf=0.01, cos_lr=True,
        warmup_epochs=5, warmup_momentum=0.8, warmup_bias_lr=0.1,
        weight_decay=0.0005, box=7.5, cls=0.5, dfl=1.5,
        label_smoothing=0.1,
        mosaic=1.0, close_mosaic=30, mixup=0.15, copy_paste=0.4,
        degrees=10.0, translate=0.1, scale=0.9, perspective=0.001,
        flipud=0.1, fliplr=0.5, erasing=0.4,
    )

BEST_PT = Path(results.save_dir) / "weights" / "best.pt"
print(f"Best model: {BEST_PT}")


## Phase 4 — Evaluate on Test Set

In [ ]:
RUN = "fracture_v3"; IMGSZ = 800
from ultralytics import YOLO
from pathlib import Path

BEST_PT = WORKING_DIR / "runs" / RUN / "weights" / "best.pt"
assert BEST_PT.exists(), f"best.pt not found at {BEST_PT}"

best = YOLO(str(BEST_PT))
(WORKING_DIR/"outputs").mkdir(exist_ok=True)

metrics = best.val(
    data=str(MERGED_YAML), split="test", device=DEVICE,
    imgsz=IMGSZ, batch=8, plots=True, save_json=True,
    project=str(WORKING_DIR/"outputs"), name="test_eval",
)
print("=" * 60)
print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print()
print("Per-class AP@50:")
for cls, ap in zip(UNIFIED_CLASSES, metrics.box.ap50):
    flag = "  << low" if ap < 0.3 else ""
    print(f"  {cls:20s}: {ap:.4f}{flag}")


## Phase 5 — Plots & Sample Predictions

In [ ]:
RUN = "fracture_v3"
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

RUN_DIR = WORKING_DIR / "runs" / RUN
OUT_DIR = WORKING_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)

for fname, title in [
    ("results.png",                     "Training Curves"),
    ("confusion_matrix_normalized.png", "Confusion Matrix"),
    ("PR_curve.png",                    "PR Curve"),
    ("F1_curve.png",                    "F1 Curve"),
]:
    p = RUN_DIR / fname
    if p.exists():
        plt.figure(figsize=(14, 8))
        plt.imshow(mpimg.imread(str(p)))
        plt.axis("off")
        plt.title(title, fontsize=14)
        plt.tight_layout()
        plt.savefig(OUT_DIR/fname, dpi=120, bbox_inches="tight")
        plt.show()
    else:
        print(f"Not found: {fname}")


In [ ]:
IMGSZ = 800
import random as _r
import matplotlib.pyplot as plt
from pathlib import Path

test_imgs = list((MERGED_DIR/"test"/"images").glob("*"))
sample    = _r.sample(test_imgs, k=min(6, len(test_imgs)))
preds     = best.predict(
    source=[str(p) for p in sample],
    imgsz=IMGSZ, device=DEVICE, conf=0.25, iou=0.45, verbose=False,
)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, r in zip(axes.flatten(), preds):
    ax.imshow(r.plot()[...,::-1])
    ax.axis("off")
    ds = {"b8":"BoneFrac","fa":"FracAtlas","gz":"GRAZ"}.get(
          Path(r.path).name[:2], "?")
    ax.set_title(f"[{ds}] {len(r.boxes) if r.boxes else 0} det(s)", fontsize=9)
plt.suptitle("Sample Predictions (conf>=0.25)", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR/"predictions.png", dpi=100, bbox_inches="tight")
plt.show()


## Phase 6 — Final Summary

In [ ]:
print("=" * 60)
print("TRAINING COMPLETE")
print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print()
print("Per-class AP@50:")
for cls, ap in zip(UNIFIED_CLASSES, metrics.box.ap50):
    print(f"  {cls:20s}: {ap:.4f}")
print()
print(f"best.pt : {BEST_PT}")
print(f"YAML    : {MERGED_YAML}")
